In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from radiomics import featureextractor
import SimpleITK as sitk
from tqdm import tqdm
from PIL import Image

try:
    input_image
except NameError:
    input_image = "test.png"
print('input_image:', input_image)


In [ ]:
# locate mask produced by py38
in_img = Path(input_image)
mask_glob = list(Path("notebooks/output_oct").glob(f"mask_{in_img.stem}*.png"))
if not mask_glob:
    raise FileNotFoundError(f"No mask found for {in_img.stem} in notebooks/output_oct")
mask_path = mask_glob[0]
print('Using mask:', mask_path)


In [ ]:
# configure radiomics extractor
extractor = featureextractor.RadiomicsFeatureExtractor()
extractor.settings.update({
    'binWidth': 25,
    'normalize': True,
    'normalizeScale': 100,
    'removeOutliers': True,
    'resampledPixelSpacing': None,
    'interpolator': sitk.sitkBSpline
})
extractor.enableAllFeatures()
print('Extractor configured')


In [ ]:
# read image and mask
img_sitk = sitk.ReadImage(str(in_img))
mask_sitk = sitk.ReadImage(str(mask_path))

if img_sitk.GetSize() != mask_sitk.GetSize():
    print('Resampling mask to match image size')
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(img_sitk)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor)
    mask_sitk = resampler.Execute(mask_sitk)

mask_arr = sitk.GetArrayFromImage(mask_sitk)
unique_labels = np.unique(mask_arr)
features_list = []
for label in unique_labels:
    if int(label) == 0:
        continue
    try:
        feats = extractor.execute(img_sitk, mask_sitk, label=int(label))
    except Exception as e:
        print(f"Skipping label {label} due to error:", e)
        continue
    feats_filtered = {k: v for k, v in feats.items() if not k.startswith('diagnostics')}
    feats_filtered['Image'] = in_img.stem
    feats_filtered['Label'] = int(label)
    features_list.append(feats_filtered)

if features_list:
    df = pd.DataFrame(features_list)
else:
    df = pd.DataFrame()

out_dir = Path('notebooks/pyradiomic_output')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f'radiomics_{in_img.stem}.csv'
df.to_csv(out_path, index=False)
print('Saved radiomics csv to', out_path)
out_path
